In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [7]:
x = df.drop('Outcome', axis=1)
y = df['Outcome']

scaler = StandardScaler()

x_scaled = pd.DataFrame(
    scaler.fit_transform(x),
    columns=x.columns,
    index=x.index
)


x = x_scaled


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


In [10]:
models = {
    'Logistic Regression': LogisticRegression(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC()
}

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f'{name} Accuracy: {accuracy*100:.4f}%')

Logistic Regression Accuracy: 75.3247%
K-Nearest Neighbors Accuracy: 68.8312%
Decision Tree Accuracy: 74.6753%
Naive Bayes Accuracy: 76.6234%
Support Vector Machine Accuracy: 72.7273%


In [12]:
models = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["liblinear", "lbfgs"]
        }
    },

    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7, 9, 11],
            "weights": ["uniform", "distance"],
            "p": [1, 2]
        }
    },

    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "criterion": ["gini", "entropy"],
            "max_depth": [None, 3, 5, 7, 10],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    },

    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {
            "var_smoothing": [1e-12, 1e-10, 1e-9, 1e-8, 1e-6]
        }
    },

    "SVM": {
        "model": SVC(),
        "params": {
            "C": [0.1, 1, 10, 100],
            "kernel": ["linear", "rbf"],
            "gamma": ["scale", "auto"]
        }
    }
}


# 2. Perform GridSearchCV for all algorithms

best_models = {}
results = []

for name, details in models.items():

    grid = GridSearchCV(
        estimator=details["model"],
        param_grid=details["params"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(x_train, y_train)

    # Store the best model
    best_models[name] = grid.best_estimator_

    # Predict on test data
    y_pred = grid.best_estimator_.predict(x_test)

    # Calculate test accuracy
    test_accuracy = accuracy_score(y_test, y_pred)

    # Save results
    results.append({
        "Algorithm": name,
        "Best CV Accuracy": grid.best_score_,
        "Test Accuracy": test_accuracy,
        "Best Parameters": grid.best_params_
    })

    print(f"\n{name}")
    print("Best Parameters:", grid.best_params_)
    print("Best CV Accuracy:", round(grid.best_score_, 4))
    print("Test Accuracy:", round(test_accuracy, 4))


# 3. Compare all algorithms

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

print("\nFinal Comparison:")
print(results_df.to_string(index=False))


# 4. Identify the model with the highest test accuracy

best_model_name = results_df.loc[0, "Algorithm"]
best_accuracy = results_df.loc[0, "Test Accuracy"]

print("\nBest Algorithm:", best_model_name)
print("Best Test Accuracy:", round(best_accuracy, 4))



Logistic Regression
Best Parameters: {'C': 10, 'solver': 'liblinear'}
Best CV Accuracy: 0.7655
Test Accuracy: 0.7532

KNN
Best Parameters: {'n_neighbors': 11, 'p': 2, 'weights': 'distance'}
Best CV Accuracy: 0.7639
Test Accuracy: 0.6883

Decision Tree
Best Parameters: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best CV Accuracy: 0.7606
Test Accuracy: 0.7662

Naive Bayes
Best Parameters: {'var_smoothing': 1e-12}
Best CV Accuracy: 0.7525
Test Accuracy: 0.7662

SVM
Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV Accuracy: 0.7655
Test Accuracy: 0.7273

Final Comparison:
          Algorithm  Best CV Accuracy  Test Accuracy                                                                         Best Parameters
      Decision Tree          0.760642       0.766234 {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 2}
        Naive Bayes          0.752472       0.766234                           

In [13]:
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Create the final Decision Tree model
final_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    min_samples_leaf=4,
    min_samples_split=2,
    random_state=42
)

# 2. Train the model
final_model.fit(x_train, y_train)

# 3. Make predictions
y_pred = final_model.predict(x_test)

# 4. Evaluate the final model
print("Final Model: Decision Tree")
print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Final Model: Decision Tree
Test Accuracy: 76.62 %

Confusion Matrix:
[[83 16]
 [20 35]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.84      0.82        99
           1       0.69      0.64      0.66        55

    accuracy                           0.77       154
   macro avg       0.75      0.74      0.74       154
weighted avg       0.76      0.77      0.76       154



In [14]:

joblib.dump(final_model, "diabetes_model.joblib")

print("Final model saved successfully!")


Final model saved successfully!


In [15]:

joblib.dump(list(x_train.columns), "diabetes_features.joblib")

print("Feature names saved successfully!")


Feature names saved successfully!


In [16]:

joblib.dump(scaler, "diabetes_scaler.joblib")

print("Scaler saved successfully!")


Scaler saved successfully!
